In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchvision.models import ResNet101_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------- Save paths ----------
save_folder = "trained_models"
os.makedirs(save_folder, exist_ok=True)

final_best_path = os.path.join(save_folder, "ResNet101_finetuned_FINAL_BEST.pth")   # ✅ ملف واحد نهائي (best)
ckpt_path       = os.path.join(save_folder, "ResNet101_finetuned_checkpoint.pth")  # ✅ للاستكمال لو قطع

# ---------- Load pretrained ResNet101 ----------
resnet101 = models.resnet101(weights=ResNet101_Weights.DEFAULT)

# ---------- Replace FC for new classes ----------
num_classes = 4
resnet101.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(resnet101.fc.in_features, num_classes)
)

resnet101 = resnet101.to(device)

# ✅ Label Smoothing
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# ---------- Helper: evaluate on val ----------
def eval_on_loader(model, loader):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss_sum += loss.item() * images.size(0)
            preds = outputs.argmax(1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
    return loss_sum / max(1, total), correct / max(1, total)

# ---------- لو best النهائي موجود ----------
if os.path.exists(final_best_path):
    resnet101.load_state_dict(torch.load(final_best_path, map_location=device))
    resnet101.eval()
    print("✅ Final BEST model loaded — ready to use!")
else:
    # =========================
    # Resume (اختياري)
    # =========================
    start_stage = 1
    start_epoch = 0
    best_val_acc = 0.0

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        resnet101.load_state_dict(ckpt["model_state"])
        best_val_acc = ckpt.get("best_val_acc", 0.0)
        start_stage = ckpt.get("stage", 1)
        start_epoch = ckpt.get("epoch", 0)
        print(f"🔄 Resuming: stage={start_stage} epoch={start_epoch} best_val_acc={best_val_acc:.4f}")

    # ============================================================
    # Stage 1: Freeze all -> Train FC فقط
    # ============================================================
    if start_stage <= 1:
        print("\n===== Stage 1: Train FC only (Frozen backbone) =====")

        for p in resnet101.parameters():
            p.requires_grad = False
        for p in resnet101.fc.parameters():
            p.requires_grad = True

        optimizer = optim.Adam(resnet101.fc.parameters(), lr=1e-4, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

        epochs_stage1 = 10
        for epoch in range(start_epoch, epochs_stage1):
            resnet101.train()
            running_loss, correct, total = 0.0, 0, 0

            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = resnet101(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item() * images.size(0)
                preds = outputs.argmax(1)
                total += labels.size(0)
                correct += (preds == labels).sum().item()

            train_loss = running_loss / max(1, total)
            train_acc  = correct / max(1, total)

            val_loss, val_acc = eval_on_loader(resnet101, val_loader)
            scheduler.step(val_loss)

            print(f"[Stage1] Epoch {epoch+1}/{epochs_stage1} | "
                  f"Train Loss {train_loss:.4f} Acc {train_acc:.4f} | "
                  f"Val Loss {val_loss:.4f} Acc {val_acc:.4f}")

            # ✅ Save best (ملف نهائي واحد)
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(resnet101.state_dict(), final_best_path)
                print(f"💾 BEST updated -> {final_best_path} (val_acc={best_val_acc:.4f})")

            # ✅ checkpoint للاستكمال
            torch.save({
                "stage": 1,
                "epoch": epoch + 1,
                "model_state": resnet101.state_dict(),
                "best_val_acc": best_val_acc
            }, ckpt_path)

        start_epoch = 0
        start_stage = 2

    # ============================================================
    # Stage 2: Fine-tune layer4 + FC
    # ============================================================
    if start_stage <= 2:
        print("\n===== Stage 2: Fine-tune layer4 + FC =====")

        for p in resnet101.parameters():
            p.requires_grad = False
        for p in resnet101.layer4.parameters():
            p.requires_grad = True
        for p in resnet101.fc.parameters():
            p.requires_grad = True

        optimizer = optim.Adam([
            {"params": resnet101.layer4.parameters(), "lr": 1e-5},
            {"params": resnet101.fc.parameters(),     "lr": 5e-5},
        ], weight_decay=1e-4)

        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

        epochs_stage2 = 15
        for epoch in range(start_epoch, epochs_stage2):
            resnet101.train()
            running_loss, correct, total = 0.0, 0, 0

            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = resnet101(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item() * images.size(0)
                preds = outputs.argmax(1)
                total += labels.size(0)
                correct += (preds == labels).sum().item()

            train_loss = running_loss / max(1, total)
            train_acc  = correct / max(1, total)

            val_loss, val_acc = eval_on_loader(resnet101, val_loader)
            scheduler.step(val_loss)

            print(f"[Stage2] Epoch {epoch+1}/{epochs_stage2} | "
                  f"Train Loss {train_loss:.4f} Acc {train_acc:.4f} | "
                  f"Val Loss {val_loss:.4f} Acc {val_acc:.4f}")

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(resnet101.state_dict(), final_best_path)
                print(f"💾 BEST updated -> {final_best_path} (val_acc={best_val_acc:.4f})")

            torch.save({
                "stage": 2,
                "epoch": epoch + 1,
                "model_state": resnet101.state_dict(),
                "best_val_acc": best_val_acc
            }, ckpt_path)

        start_epoch = 0
        start_stage = 3

    # ============================================================
    # Stage 3: Fine-tune layer3 + layer4 + FC (Progressive Unfreezing)
    # ============================================================
    if start_stage <= 3:
        print("\n===== Stage 3: Fine-tune layer3 + layer4 + FC (Best boost) =====")

        for p in resnet101.parameters():
            p.requires_grad = False
        for p in resnet101.layer3.parameters():
            p.requires_grad = True
        for p in resnet101.layer4.parameters():
            p.requires_grad = True
        for p in resnet101.fc.parameters():
            p.requires_grad = True

        optimizer = optim.Adam([
            {"params": resnet101.layer3.parameters(), "lr": 5e-6},
            {"params": resnet101.layer4.parameters(), "lr": 1e-5},
            {"params": resnet101.fc.parameters(),     "lr": 5e-5},
        ], weight_decay=1e-4)

        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

        epochs_stage3 = 10
        for epoch in range(start_epoch, epochs_stage3):
            resnet101.train()
            running_loss, correct, total = 0.0, 0, 0

            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = resnet101(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item() * images.size(0)
                preds = outputs.argmax(1)
                total += labels.size(0)
                correct += (preds == labels).sum().item()

            train_loss = running_loss / max(1, total)
            train_acc  = correct / max(1, total)

            val_loss, val_acc = eval_on_loader(resnet101, val_loader)
            scheduler.step(val_loss)

            print(f"[Stage3] Epoch {epoch+1}/{epochs_stage3} | "
                  f"Train Loss {train_loss:.4f} Acc {train_acc:.4f} | "
                  f"Val Loss {val_loss:.4f} Acc {val_acc:.4f}")

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(resnet101.state_dict(), final_best_path)
                print(f"💾 BEST updated -> {final_best_path} (val_acc={best_val_acc:.4f})")

            torch.save({
                "stage": 3,
                "epoch": epoch + 1,
                "model_state": resnet101.state_dict(),
                "best_val_acc": best_val_acc
            }, ckpt_path)

    # في الآخر حملي أفضل نسخة
    resnet101.load_state_dict(torch.load(final_best_path, map_location=device))
    resnet101.eval()
    print(f"\n✅ Done! Best model saved at: {final_best_path} (best_val_acc={best_val_acc:.4f})")


In [ ]:
import os
import torch

def evaluate_or_load(
    model,
    loader,
    set_name="Test",
    model_name="ResNet101",
    save_dir="results"
):
    os.makedirs(save_dir, exist_ok=True)

    save_path = os.path.join(save_dir, f"{model_name}_{set_name}_evaluation.txt")

    # =========================
    # ✅ لو التقييم محفوظ → اقريه
    # =========================
    if os.path.exists(save_path):
        print(f"📂 Found saved evaluation → Loading from:\n{save_path}\n")
        with open(save_path, "r") as f:
            print(f.read())

        # نرجّع None لأننا مستخدمناش الحساب
        return None, None

    # =========================
    # ❌ مش موجود → احسبي و احفظي
    # =========================
    print("🚀 No saved evaluation found → Running evaluation...")

    model.eval()
    correct = 0
    total = 0
    running_loss = 0.0
    criterion = torch.nn.CrossEntropyLoss()

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = running_loss / total
    accuracy = correct / total

    # =========================
    # 💾 حفظ التقييم
    # =========================
    with open(save_path, "w", encoding="utf-8") as f:
        f.write(f"{set_name} Evaluation ({model_name})\n")
        f.write(f"Loss: {avg_loss:.4f}\n")
        f.write(f"Accuracy: {accuracy:.4f}\n")

    print(f"\n✅ Evaluation saved to:\n{save_path}")
    print(f"Loss: {avg_loss:.4f}")
    print(f"Accuracy: {accuracy:.4f}")

    return avg_loss, accuracy


In [ ]:
test_loss, test_acc = evaluate_or_load(
    resnet101,
    test_loader,
    set_name="Test",
    model_name="ResNet101"
)


In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

# ---------------- التحويلات زي اللي اتعملت للـ val/test ----------------
val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ---------------- Load class names ----------------
class_names = ["glass", "metal", "plastic", "wood"]
  # ['glass', 'metal', 'plastic', 'wood'

# ---------------- Load image ----------------
image_path = r"C:\Users\user\OneDrive - Helwan National University\Desktop\final aiiiiiiiiiiiiiii\dataset_processed\test\plastic\plastic_000029.jpg"

image = Image.open(image_path).convert("RGB")
input_tensor = val_test_transform(image).unsqueeze(0).to(device)  # 1xCxHxW

# ---------------- Prediction ----------------
resnet101.eval()
with torch.no_grad():
    outputs = resnet101(input_tensor)
    probs = torch.softmax(outputs, dim=1)
    conf, pred = torch.max(probs, 1)

# ---------------- Display ----------------
plt.imshow(image)
plt.axis('off')
plt.show()

print(f"Predicted Material: {class_names[pred.item()]}")
print(f"Confidence: {conf.item():.4f}")


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import transforms, datasets, models
from torchvision.models import ResNet101_Weights
from sklearn.metrics import precision_score, recall_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# =========================
# 1) Device
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================
# 2) Paths
# =========================
data_dir  = r"dataset_processed/test"
ckpt_path = r"C:\Users\user\OneDrive - Helwan National University\Desktop\final aiiiiiiiiiiiiiii\trained_models\ResNet101_finetuned_FINAL_BEST.pth"

save_dir = "results"
os.makedirs(save_dir, exist_ok=True)

model_name = "ResNet101"
set_name   = "Test"

pr_path  = os.path.join(save_dir, f"{model_name}_{set_name}_precision_recall.txt")
cm_png   = os.path.join(save_dir, f"{model_name}_{set_name}_confusion_matrix.png")
cm_csv   = os.path.join(save_dir, f"{model_name}_{set_name}_confusion_matrix.csv")

# =========================
# 3) Dataset
# =========================
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

test_dataset = datasets.ImageFolder(data_dir, transform=transform)
test_loader  = torch.utils.data.DataLoader(test_dataset, batch_size=16, shuffle=False)
class_names  = test_dataset.classes

print("Classes:", class_names)
print("Test samples:", len(test_dataset))

# =========================
# 4) Load ResNet101
# =========================
model = models.resnet101(weights=ResNet101_Weights.DEFAULT)
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, len(class_names))
)

state = torch.load(ckpt_path, map_location=device)

# handle different save formats just in case
if isinstance(state, dict) and "model_state" in state:
    state = state["model_state"]
elif isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]

# remove "module." prefix if present
if isinstance(state, dict) and any(k.startswith("module.") for k in state.keys()):
    state = {k.replace("module.", ""): v for k, v in state.items()}

model.load_state_dict(state, strict=True)
model = model.to(device)
model.eval()
print("✅ ResNet101 loaded.")

# =========================
# 5) Collect all_preds / all_labels (REQUIRED!)
# =========================
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

print("✅ Predictions collected:", len(all_labels))

# =========================
# 6) Precision / Recall (Load if exists else compute+save)
# =========================
if os.path.exists(pr_path):
    print(f"\n📂 Found saved Precision/Recall → Loading:\n{pr_path}\n")
    with open(pr_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    precision_per_class = precision_score(all_labels, all_preds, average=None, zero_division=0)
    recall_per_class    = recall_score(all_labels, all_preds, average=None, zero_division=0)

    lines = []
    lines.append(f"Precision & Recall per class ({model_name}) - {set_name}\n")
    for i, cls in enumerate(class_names):
        lines.append(f"{cls}:")
        lines.append(f"  Precision = {precision_per_class[i]:.4f}")
        lines.append(f"  Recall    = {recall_per_class[i]:.4f}\n")

    text = "\n".join(lines)

    with open(pr_path, "w", encoding="utf-8") as f:
        f.write(text)

    print("\n" + text)
    print(f"✅ Precision/Recall saved to:\n{pr_path}")

# =========================
# 7) Confusion Matrix (Load if exists else compute+save)
# =========================
def show_cm(df_cm, title):
    plt.figure(figsize=(8,6))
    sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.tight_layout()
    plt.show()

if os.path.exists(cm_png) and os.path.exists(cm_csv):
    print(f"\n📂 Found saved Confusion Matrix → Loading:\n{cm_csv}\n")
    df_cm = pd.read_csv(cm_csv, index_col=0)
    show_cm(df_cm, f"Confusion Matrix ({model_name}) - {set_name}")
else:
    cm = confusion_matrix(all_labels, all_preds)
    df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)

    print("\n📌 Confusion Matrix (Raw):")
    print(cm)

    df_cm.to_csv(cm_csv)

    plt.figure(figsize=(8,6))
    sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix ({model_name}) - {set_name}")
    plt.tight_layout()
    plt.savefig(cm_png, dpi=300)
    plt.show()

    print("\n✅ Confusion Matrix saved to:")
    print(cm_png)
    print(cm_csv)


In [ ]:
# ====== 1) الاستيراد ======
import torch
from torchvision import transforms, datasets, models
import torch.nn as nn
from torchvision.models import ResNet101_Weights
from PIL import Image
import torch.nn.functional as F
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ====== 2) الجهاز ======
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ====== 3) الـ transform و الـ dataset ======
data_dir = r"C:\Users\user\OneDrive - Helwan National University\Desktop\final aiiiiiiiiiiiiiii\dataset_processed\test"

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

test_dataset = datasets.ImageFolder(data_dir, transform=transform)
class_names = test_dataset.classes
classes = class_names  # عشان نستخدمها في العنوان بتاع Grad-CAM

print("Classes:", classes)

# ====== 4) تحميل نموذج ResNet101 المدرب ======
model = models.resnet101(weights=ResNet101_Weights.DEFAULT)
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, len(class_names))
)

model.load_state_dict(torch.load('trained_models\ResNet101_finetuned_FINAL_BEST.pth', map_location=device))
model = model.to(device)
model.eval()
print("Model loaded.")

# ====== 5) تعريف دالة Grad-CAM ======
def show_gradcam_resnet(model, input_tensor, target_class=None, device='cuda'):
    model.eval()
    input_tensor = input_tensor.to(device)

    activations = None
    gradients = None

    def forward_hook(module, input, output):
        nonlocal activations
        activations = output

    def backward_hook(module, grad_in, grad_out):
        nonlocal gradients
        gradients = grad_out[0]

    # -------- اختر آخر طبقة في ResNet101 --------
    target_layer = model.layer4[-1]
    fh = target_layer.register_forward_hook(forward_hook)
    bh = target_layer.register_backward_hook(backward_hook)

    # -------- Forward --------
    output = model(input_tensor)
    if target_class is None:
        target_class = output.argmax(dim=1).item()

    # -------- Backward --------
    model.zero_grad()
    loss = output[0, target_class]
    loss.backward()

    # -------- GradCAM --------
    pooled_grads = torch.mean(gradients, dim=[0, 2, 3])
    acts = activations.clone()

    for i in range(acts.shape[1]):
        acts[0, i, :, :] *= pooled_grads[i]

    heatmap = torch.sum(acts, dim=1).squeeze().cpu().detach().numpy()
    heatmap = np.maximum(heatmap, 0)
    heatmap = heatmap / (np.max(heatmap) + 1e-8)

    # -------- Superimpose --------
    img = input_tensor[0].cpu().permute(1,2,0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_color = np.float32(heatmap_color) / 255

    overlay = heatmap_color + img
    overlay /= overlay.max()

    plt.figure(figsize=(6,6))
    plt.imshow(overlay)
    plt.axis('off')
    plt.title(f"Grad-CAM for class {classes[target_class]}")
    plt.show()

    fh.remove()
    bh.remove()

# ====== 6) اختار صورة من الـ test dataset (أسهل حاجة) ======
# بدل ما ندوّخ نفسنا بمسار صورة، هناخد أول صورة من الـ dataset
img, label = test_dataset[700]
input_tensor = img.unsqueeze(0)  # [1, 3, 224, 224]

# ====== 7) Prediction على الصورة ======
outputs = model(input_tensor.to(device))
_, pred = torch.max(outputs, 1)
print("Predicted class:", classes[pred.item()])

# ====== 8) تشغيل Grad-CAM ======
show_gradcam_resnet(model, input_tensor, target_class=pred.item(), device=device)
